In [ ]:
from pathlib import Path
import numpy as np
import torch
from tqdm import tqdm

from IPPy import operators
from IPPy.solvers import ChambollePockTpVConstrained
from IPPy.utilities import normalize, save_image

from utils.data_utils import load_normalized_image

# Input
DATASET_DIRS = {
    "train": Path("../dataset_resized/train_resized"),
    "validation": Path("../dataset_resized/validation_resized"),
    "test": Path("../dataset_resized/test_resized"),
}
SINOGRAMS_DIR = Path("../sinograms")

# Output
TV_DIR = Path("../reconstructions/tv")
TV_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (256, 256)
NOISE_LEVEL = 0.005

ANGLE_CONFIGS = {
    90: np.linspace(-45, 45, 90),
    45: np.linspace(-45, 45, 45),
    30: np.linspace(-30, 30, 32)[1:-1],
    15: np.linspace(-30, 30, 17)[1:-1],
}

# Parametri trovati nel tuning (copiati dai blocchi precedenti)
BEST_LAMBDA = {90:  0.05, 45: 0.05, 30: 0.1, 15: 0.1}       # <- sostituisci con i tuoi valori reali (GIA FATTO IO)
BEST_MAXITER = {90:  0.05, 45:  0.05, 30: 0.1, 15: 0.1}          # <- sostituisci con i tuoi valori reali (GIA FATTO IO)

PROJECTORS = {
    n_angles: operators.CTProjector(
        img_shape=IMG_SIZE,
        angles=np.deg2rad(angles),
        geometry="parallel",
        force_cpu=True,
    )
    for n_angles, angles in ANGLE_CONFIGS.items()
}


def reconstruct_tv_split(split_name: str, gt_root: Path):
    for n_angles, K in PROJECTORS.items():
        lambda_tv = BEST_LAMBDA[n_angles]
        maxiter = BEST_MAXITER[n_angles]

        sino_paths = sorted((SINOGRAMS_DIR / split_name / str(n_angles)).rglob("*.npy"))

        for sino_path in tqdm(sino_paths, desc=f"TV [{split_name}] {n_angles} angoli"):
            rel_path = sino_path.relative_to(SINOGRAMS_DIR / split_name / str(n_angles))
            out_path = (TV_DIR / split_name / str(n_angles) / rel_path).with_suffix(".png")
            out_path.parent.mkdir(parents=True, exist_ok=True)

            if out_path.exists():
                continue

            gt_path = (gt_root / rel_path).with_suffix(".png")

            sinogram = np.load(sino_path)
            y_delta = torch.from_numpy(sinogram).float().unsqueeze(0).unsqueeze(0)

            gt = load_normalized_image(gt_path)
            x_true = torch.from_numpy(gt).float().unsqueeze(0).unsqueeze(0)

            epsilon = NOISE_LEVEL * torch.norm(y_delta)
            solver = ChambollePockTpVConstrained(K)

            x_sol, _ = solver(
                y_delta,
                epsilon=epsilon,
                lmbda=lambda_tv,
                x_true=x_true,
                starting_point=torch.zeros_like(x_true),
                maxiter=maxiter,
                p=1,
                verbose=False,
            )

            x_sol = x_sol.detach()
            save_image(normalize(x_sol), out_path)


for split_name, gt_root in DATASET_DIRS.items():
    reconstruct_tv_split(split_name, gt_root)

print("Ricostruzioni TV completate su tutti gli split.")